In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.dataset import ImageDataset
from torch.utils.data import DataLoader

annotations_file_trainval = Path("../data/preprocessed/trainval/annotations.csv")
img_dir_trainval = Path("../data/preprocessed/trainval/Images")

trainval_dataset = ImageDataset(annotations_file_trainval, img_dir_trainval)
trainval_dl = DataLoader(trainval_dataset, batch_size=32, shuffle=True)

In [3]:
from src.model import Model

model = Model()

In [4]:
from src.configs import S, B, C, IDX_TO_CLASS
from src.utils import convert_xywh_coords

def decode_preds(preds_batch):
    decoded_preds = []

    for pred in preds_batch:
        pred = pred.reshape((S, S, C + B * 5))
        objects = []

        for i in range(S):
            for j in range(S):
                pred_cell = pred[i][j]

                pred_class_idx = pred_cell[:20].argmax().item()
                pred_class_prob = pred_cell[pred_class_idx].item()
                pred_class = IDX_TO_CLASS[pred_class_idx]
                
                pred_1_confidence = (pred_cell[24].item() * \
                                     pred_class_prob,)
                pred_2_confidence = (pred_cell[29].item() * \
                                     pred_class_prob,)

                bbox_1 = convert_xywh_coords(pred_cell[20:24], i, j, False, True)
                bbox_2 = convert_xywh_coords(pred_cell[25:29], i, j, False, True)
                
                objects.append((pred_class,) + pred_1_confidence + bbox_1)
                objects.append((pred_class,) + pred_2_confidence + bbox_2)

        decoded_preds.append(objects)

    return decoded_preds

In [5]:
X_batch, y_batch = next(iter(trainval_dl))

X_batch.shape, y_batch.shape

(torch.Size([32, 3, 224, 224]), torch.Size([32, 7, 7, 30]))

In [6]:
preds = model(X_batch)
preds.shape

torch.Size([32, 1470])

In [7]:
preds = preds.reshape((preds.shape[0], S, S, B * 5 + C))
preds.shape

torch.Size([32, 7, 7, 30])

In [8]:
decoded_preds = decode_preds(preds)
len(decoded_preds), len(decoded_preds[0])

(32, 98)

In [9]:
CONFIDENCE_THRESHOLD = 0.375
from operator import itemgetter

def filter_sort(decoded_preds):
    sorted_preds = []

    # 1. filter and  by class
    for image in decoded_preds:
        valid_preds = {}
        for pred in image:
            if pred[1] > CONFIDENCE_THRESHOLD:
                class_name = pred[0]
                if class_name in valid_preds:
                    valid_preds[class_name].append(pred)
                else:
                    valid_preds[class_name] = [pred]
    
        sorted_preds.append(valid_preds)

    # 2. sort each class by confidence score 
    for image in sorted_preds:
        for class_name in image:
            image[class_name].sort(key=itemgetter(1), reverse=True)
    
    return sorted_preds

In [17]:
sorted_preds = filter_sort(decoded_preds)
sorted_preds

[{'diningtable': [('diningtable',
    0.42834684432120795,
    94.57119750976562,
    72.0880126953125,
    150.29067993164062,
    70.38554382324219)]},
 {},
 {'dog': [('dog',
    0.4091469681466755,
    96.38333892822266,
    68.61160278320312,
    92.1452407836914,
    -72.78907775878906)],
  'sofa': [('sofa',
    0.4716690963724517,
    115.46290588378906,
    84.48628997802734,
    122.09098815917969,
    133.41751098632812)]},
 {'diningtable': [('diningtable',
    0.38613386137898686,
    117.59696960449219,
    122.66044616699219,
    155.3568878173828,
    110.06806945800781)],
  'boat': [('boat',
    0.39912335872222116,
    222.41847229003906,
    201.84837341308594,
    180.4741973876953,
    183.95506286621094)]},
 {},
 {'bottle': [('bottle',
    0.912612280530027,
    120.26898956298828,
    146.9519805908203,
    136.6679229736328,
    95.08108520507812),
   ('bottle',
    0.4836878923107122,
    181.0317840576172,
    63.63353729248047,
    -40.214691162109375,
    -48.2

In [20]:
sorted_preds[5]['bottle']

[('bottle',
  0.912612280530027,
  120.26898956298828,
  146.9519805908203,
  136.6679229736328,
  95.08108520507812),
 ('bottle',
  0.4836878923107122,
  181.0317840576172,
  63.63353729248047,
  -40.214691162109375,
  -48.272010803222656)]

In [21]:
final_preds = []
    
for image in sorted_preds:
    final_img_preds = {}

    for class_name, item in image.items():
        highest_conf = item.pop(0)
        final_img_preds[class_name] = [highest_conf]

    final_preds.append(final_img_preds)

([('bottle',
   0.4836878923107122,
   181.0317840576172,
   63.63353729248047,
   -40.214691162109375,
   -48.272010803222656)],
 [{'diningtable': [('diningtable',
     0.42834684432120795,
     94.57119750976562,
     72.0880126953125,
     150.29067993164062,
     70.38554382324219)]},
  {},
  {'dog': [('dog',
     0.4091469681466755,
     96.38333892822266,
     68.61160278320312,
     92.1452407836914,
     -72.78907775878906)],
   'sofa': [('sofa',
     0.4716690963724517,
     115.46290588378906,
     84.48628997802734,
     122.09098815917969,
     133.41751098632812)]},
  {'diningtable': [('diningtable',
     0.38613386137898686,
     117.59696960449219,
     122.66044616699219,
     155.3568878173828,
     110.06806945800781)],
   'boat': [('boat',
     0.39912335872222116,
     222.41847229003906,
     201.84837341308594,
     180.4741973876953,
     183.95506286621094)]},
  {},
  {'bottle': [('bottle',
     0.912612280530027,
     120.26898956298828,
     146.9519805908203,

In [24]:
final_preds

[{'diningtable': [('diningtable',
    0.42834684432120795,
    94.57119750976562,
    72.0880126953125,
    150.29067993164062,
    70.38554382324219)]},
 {},
 {'dog': [('dog',
    0.4091469681466755,
    96.38333892822266,
    68.61160278320312,
    92.1452407836914,
    -72.78907775878906)],
  'sofa': [('sofa',
    0.4716690963724517,
    115.46290588378906,
    84.48628997802734,
    122.09098815917969,
    133.41751098632812)]},
 {'diningtable': [('diningtable',
    0.38613386137898686,
    117.59696960449219,
    122.66044616699219,
    155.3568878173828,
    110.06806945800781)],
  'boat': [('boat',
    0.39912335872222116,
    222.41847229003906,
    201.84837341308594,
    180.4741973876953,
    183.95506286621094)]},
 {},
 {'bottle': [('bottle',
    0.912612280530027,
    120.26898956298828,
    146.9519805908203,
    136.6679229736328,
    95.08108520507812)],
  'dog': [('dog',
    0.3796550058319781,
    190.9224853515625,
    -28.675678253173828,
    136.8143310546875,
    

In [ ]:
from utils import IoU

def NMS(preds_batch):
    # 1. decode batch of predictions
    decoded_preds = decode_preds(preds_batch)

    # 2. filter, group, and sort the decoded predictions
    sorted_preds = filter_group_sort_preds(decoded_preds)

    # 3. perform Non-Maximum Suppression
    final_preds = []
    
    for image in sorted_preds:
        final_img_preds = {}

        for class_name, items in image.items():
            final_img_preds[class_name] = []

            while items: 
                highest_conf = items.pop(0)
                final_img_preds.append(highest_conf)

                for item in items:
                    IoU_ = IoU(

            
            
        